# Superhero Intelligence System
**Can six ability scores tell a hero from a villain?**

Sections: 2 Data Preparation · Task 1 Classification · Task 2 Clustering · Task 3 Neural Network

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

SCORES = ["intelligence_score", "strength_score", "speed_score",
          "durability_score", "power_score", "combat_score"]
COLORS = {"hero": "#1F5FBF", "villain": "#D62839", "baseline": "#9E9E9E",
          "tree": "#2E7D32", "knn": "#F9A825", "netA": "#6A1B9A", "netB": "#00838F"}

## Section 2: Data Preparation

### Step 1: Inspect (T1)

In [2]:
train_raw = pd.read_csv("data/train.csv")
test_raw = pd.read_csv("data/test.csv")

def inspect(df, label):
    print(f"== {label}: shape {df.shape}")
    use = SCORES + ["alignment", "name"]
    num = df[SCORES].apply(pd.to_numeric, errors="coerce")
    t = pd.DataFrame({
        "dtype": df[use].dtypes.astype(str),
        "missing": df[use].isna().sum(),
        "zeros": [(num[c] == 0).sum() if c in SCORES else np.nan for c in use],
    })
    # blanks/non-numeric that become NaN + zeros that become NaN
    t["-> NaN (total)"] = [num[c].isna().sum() + (num[c] == 0).sum() if c in SCORES else np.nan for c in use]
    display(t)

print("Columns:", list(train_raw.columns))
inspect(train_raw, "train.csv"); inspect(test_raw, "test.csv")

Columns: ['name', 'real_name', 'full_name', 'overall_score', 'history_text', 'powers_text', 'intelligence_score', 'strength_score', 'speed_score', 'durability_score', 'power_score', 'combat_score', 'superpowers', 'alter_egos', 'aliases', 'place_of_birth', 'first_appearance', 'creator', 'alignment', 'occupation', 'base', 'teams', 'relatives', 'gender', 'type_race', 'height', 'weight', 'eye_color', 'hair_color', 'skin_color', 'img', 'has_electrokinesis', 'has_energy_constructs', 'has_mind_control_resistance', 'has_matter_manipulation', 'has_telepathy_resistance', 'has_mind_control', 'has_enhanced_hearing', 'has_dimensional_travel', 'has_element_control', 'has_size_changing', 'has_fire_resistance', 'has_fire_control', 'has_dexterity', 'has_reality_warping', 'has_illusions', 'has_energy_beams', 'has_peak_human_condition', 'has_shapeshifting', 'has_heat_resistance', 'has_jump', 'has_self-sustenance', 'has_energy_absorption', 'has_cold_resistance', 'has_magic', 'has_telekinesis', 'has_toxin_

,dtype,missing,zeros,-> NaN (total)
intelligence_score,int64,0,85.0,85.0
strength_score,int64,0,91.0,91.0
speed_score,int64,0,86.0,86.0
durability_score,int64,0,87.0,87.0
power_score,int64,0,57.0,57.0
combat_score,int64,0,85.0,85.0
alignment,str,66,NaN,NaN
name,str,0,NaN,NaN


== test.csv: shape (290, 83)


,dtype,missing,zeros,-> NaN (total)
intelligence_score,int64,0,20.0,20.0
strength_score,int64,0,22.0,22.0
speed_score,int64,0,20.0,20.0
durability_score,int64,0,20.0,20.0
power_score,int64,0,14.0,14.0
combat_score,int64,0,20.0,20.0
alignment,str,14,NaN,NaN
name,str,0,NaN,NaN


### Steps 2-4: duplicates, mark missing, drop weak rows (T2)

In [3]:
n = {"Loaded": (len(train_raw), len(test_raw))}
cols = ["name", "alignment"] + SCORES
train, test = train_raw[cols].copy(), test_raw[cols].copy()

# Step 2: duplicates
train = train.drop_duplicates("name", keep="first")
test = test.drop_duplicates("name", keep="first")
train = train[~train["name"].isin(test["name"])]
n["After step 2 (duplicates)"] = (len(train), len(test))

# Step 3: blanks and 0 -> NaN
def mark_missing(df):
    df = df.copy()
    df[SCORES] = df[SCORES].apply(pd.to_numeric, errors="coerce").replace(0, np.nan)
    return df
train, test = mark_missing(train), mark_missing(test)

# Step 4: drop rows with 3+ missing scores
train = train[train[SCORES].isna().sum(axis=1) < 3]
test = test[test[SCORES].isna().sum(axis=1) < 3]
n["After step 4 (weak rows)"] = (len(train), len(test))

### Clean Set A (clustering) and Clean Set B (classification / NN)

In [4]:
set_a = train.reset_index(drop=True)                      # Task 2: all characters, train only

def set_b(df):
    df = df[df["alignment"].isin(["Good", "Bad"])].copy()
    df["target"] = df["alignment"].map({"Good": 0, "Bad": 1})
    return df.reset_index(drop=True)
train_b, test_b = set_b(train), set_b(test)               # Tasks 1 & 3

n["Clean Set A"] = (len(set_a), "not used")
n["Clean Set B"] = (len(train_b), len(test_b))
T2 = pd.DataFrame(n, index=["train.csv", "test.csv"]).T
display(T2)
print("Removed from Set B (Neutral/blank alignment): train", len(train) - len(train_b),
      "| test", len(test) - len(test_b))
print("Alignment values in cleaned train:", train["alignment"].value_counts(dropna=False).to_dict())

,train.csv,test.csv
Loaded,1158,290
After step 2 (duplicates),1158,290
After step 4 (weak rows),1072,270
Clean Set A,1072,not used
Clean Set B,916,225


Removed from Set B (Neutral/blank alignment): train 156 | test 45
Alignment values in cleaned train: {'Good': 588, 'Bad': 328, 'Neutral': 93, nan: 63}


## Task 1: Classification (25 marks)
_Baseline, Decision Tree, KNN on Clean Set B. Test once at the end._

### Class balance (T3, G1)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support, accuracy_score, f1_score)
from matplotlib.colors import LinearSegmentedColormap

X_tr, y_tr = train_b[SCORES], train_b["target"]
X_te, y_te = test_b[SCORES], test_b["target"]

counts = pd.DataFrame({"train.csv": train_b["alignment"].value_counts(),
                       "test.csv": test_b["alignment"].value_counts()}).loc[["Good", "Bad"]]
T3 = counts.join((counts / counts.sum() * 100).round(1).add_suffix(" %"))
print("T3: Class counts and % in each file"); display(T3)

fig, ax = plt.subplots(figsize=(6, 4))
w = 0.35
for j, (cls, col) in enumerate([("Good", COLORS["hero"]), ("Bad", COLORS["villain"])]):
    bars = ax.bar(np.arange(2) + (j - 0.5) * w, counts.loc[cls], w, color=col, label=f"{cls} ({'Hero' if cls=='Good' else 'Villain'})")
    for b, v, tot in zip(bars, counts.loc[cls], counts.sum()):
        ax.text(b.get_x() + w / 2, b.get_height() + 5, f"{v}\n{v/tot:.1%}", ha="center", fontsize=9)
ax.set_xticks(range(2)); ax.set_xticklabels(counts.columns)
ax.set_ylim(0, counts.values.max() * 1.2)
ax.set_xlabel("File"); ax.set_ylabel("Characters"); ax.set_title("G1: Class counts (Clean Set B)"); ax.legend()
plt.show()

**G1:** Heroes outnumber villains in both files (64/36 in train, 61/39 in test). The imbalance is why macro-F1 is the main metric.

### Models and grid search (T4)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
median = lambda: SimpleImputer(strategy="median")

baseline = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)

tree_gs = GridSearchCV(
    Pipeline([("imputer", median()), ("model", DecisionTreeClassifier(criterion="gini", random_state=SEED))]),
    {"model__max_depth": [3, 5, 7, 10, None], "model__min_samples_leaf": [1, 5, 10]},
    cv=cv, scoring="f1_macro", refit=True)
knn_gs = GridSearchCV(
    Pipeline([("imputer", median()), ("scaler", StandardScaler()),
              ("model", KNeighborsClassifier(metric="euclidean"))]),
    {"model__n_neighbors": [3, 5, 7, 9, 11, 15], "model__weights": ["uniform", "distance"]},
    cv=cv, scoring="f1_macro", refit=True)
tree_gs.fit(X_tr, y_tr); knn_gs.fit(X_tr, y_tr)

T4 = pd.DataFrame({
    "Model": ["Decision Tree", "KNN"],
    "Best settings": [{k.split("__")[1]: v for k, v in g.best_params_.items()} for g in (tree_gs, knn_gs)],
    "Best CV macro-F1": [round(g.best_score_, 3) for g in (tree_gs, knn_gs)]})
print("T4: Best settings and best CV macro-F1"); display(T4)

### Test once (T5)

In [ ]:
models = {"Baseline": baseline, "Decision Tree": tree_gs.best_estimator_, "KNN": knn_gs.best_estimator_}
MODEL_COLORS = {"Baseline": COLORS["baseline"], "Decision Tree": COLORS["tree"], "KNN": COLORS["knn"]}

def evaluate(m):
    p = m.predict(X_te)
    (ph, pv), (rh, rv), (fh, fv), _ = precision_recall_fscore_support(y_te, p, labels=[0, 1], zero_division=0)
    return {"Train Macro-F1": f1_score(y_tr, m.predict(X_tr), average="macro"),
            "Accuracy": accuracy_score(y_te, p),
            "Precision Hero": ph, "Recall Hero": rh, "F1 Hero": fh,
            "Precision Villain": pv, "Recall Villain": rv, "F1 Villain": fv,
            "Macro-F1": f1_score(y_te, p, average="macro"),
            "Macro-Precision": (ph + pv) / 2, "Macro-Recall": (rh + rv) / 2}

T5_full = pd.DataFrame({k: evaluate(m) for k, m in models.items()}).T
T5 = T5_full.drop(columns=["Macro-Precision", "Macro-Recall"])
print("T5: Task 1 results (test.csv, except Train Macro-F1)"); display(T5.round(3))
for k, m in models.items():
    print(f"--- {k}"); print(classification_report(y_te, m.predict(X_te), target_names=["Hero", "Villain"], digits=3, zero_division=0))

### Confusion matrices (G2, G3)

In [ ]:
def plot_cm(ax, model, name, color):
    cm = confusion_matrix(y_te, model.predict(X_te), labels=[0, 1])
    ax.imshow(cm, cmap=LinearSegmentedColormap.from_list("c", ["white", color]))
    for (r, c), v in np.ndenumerate(cm):
        ax.text(c, r, v, ha="center", va="center", fontsize=14, color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Good", "Bad"]); ax.set_yticks([0, 1]); ax.set_yticklabels(["Good", "Bad"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(name)
    return cm

fig, ax = plt.subplots(figsize=(4, 3.6)); cm_tree = plot_cm(ax, models["Decision Tree"], "G2: Confusion matrix, Decision Tree (test)", COLORS["tree"]); plt.show()
fig, ax = plt.subplots(figsize=(4, 3.6)); cm_knn = plot_cm(ax, models["KNN"], "G3: Confusion matrix, KNN (test)", COLORS["knn"]); plt.show()

### Model comparison (G4)

In [ ]:
metrics = ["Accuracy", "Macro-Precision", "Macro-Recall", "Macro-F1"]
fig, ax = plt.subplots(figsize=(8, 4)); w = 0.25
for j, name in enumerate(models):
    vals = T5_full.loc[name, metrics].astype(float)
    bars = ax.bar(np.arange(4) + (j - 1) * w, vals, w, color=MODEL_COLORS[name], label=name)
    for b, v in zip(bars, vals): ax.text(b.get_x() + w / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
ax.set_xticks(range(4)); ax.set_xticklabels(metrics); ax.set_ylim(0, 1)
ax.set_xlabel("Metric"); ax.set_ylabel("Score (test.csv)"); ax.set_title("G4: Model comparison"); ax.legend()
plt.show()

### Winner by the rules

In [ ]:
t, k, b = (T5_full.loc[n] for n in ["Decision Tree", "KNN", "Baseline"])
if abs(t["Macro-F1"] - k["Macro-F1"]) < 0.01:
    winner = "Decision Tree" if t["Recall Villain"] >= k["Recall Villain"] else "KNN"; rule = "tie-break on Villain recall"
else:
    winner = "Decision Tree" if t["Macro-F1"] > k["Macro-F1"] else "KNN"; rule = "highest test Macro-F1"
print(f"Winner: {winner} ({rule})")
for n in ["Decision Tree", "KNN"]:
    r = T5_full.loc[n]
    print(f"{n}: beats baseline by {r['Macro-F1'] - b['Macro-F1']:+.3f} | train-test gap {r['Train Macro-F1'] - r['Macro-F1']:.3f}"
          f" -> {'overfits' if r['Train Macro-F1'] - r['Macro-F1'] > 0.10 else 'no overfit flag'}")

### Task 1 answers
**Q1.** KNN wins: test Macro-F1 0.526 vs 0.408 for the Decision Tree (gap well above 0.01, so no tie-break). Best settings: KNN k=3, uniform weights; Tree max_depth=5, min_samples_leaf=10.

**Q2.** Both beat the Baseline (0.378) on Macro-F1, but by very different margins: KNN by +0.148, the Decision Tree by only +0.030. The Baseline's 0.609 accuracy is just the Hero share of test; it never finds a villain. Both models have lower accuracy than the Baseline (0.573 and 0.560), which is the price of trying to catch villains.

**Q3.** Villains are harder. Decision Tree: 6 of 88 villains caught (recall 0.068), 82 missed. KNN: 29 of 88 caught (recall 0.330), 59 missed. Heroes are recalled far better (0.876 and 0.730). The models lean towards predicting Hero because it is the majority class.

**Q4.** Both overfit by the rule (train − test Macro-F1 > 0.10): Tree 0.563 − 0.408 = 0.155, KNN 0.739 − 0.526 = 0.213. KNN with k=3 memorises the training data (its training score is inflated because each point is its own nearest neighbour). Note the best CV macro-F1 scores (0.540 and 0.517) are close to the test scores, so the weak performance is real, not just an overfitting artefact: the 6 scores carry little signal about alignment.


## Task 2: Clustering (25 marks)
_K-Means and Agglomerative on Clean Set A. `alignment` only looked at after clustering._

## Task 3: Neural Network (25 marks)
_Network A (6-8-1) and Network B (6-16-8-1). Final ranking of all 5 models._